In [1]:
import pandas as pd
import numpy as np
import plotly

### Load single-cell data

In [2]:
%%time
df = pd.read_pickle('/Users/dayn/data/macrohet_mac/verified_df.pkl')


CPU times: user 639 ms, sys: 285 ms, total: 924 ms
Wall time: 1.04 s


In [10]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# --- 1. Dimensions & Typography Configuration ---
# 210mm is approximately 794 pixels at 96 DPI
A4_WIDTH_PX = 794 
FIG_HEIGHT_PX = 550 # Adjusted for a balanced aspect ratio on A4

# Font Sizes
FONT_TITLE = 12
FONT_SUBPLOT = 9
FONT_NODES = 7

# Viridis Palette
c_infected = '#440154'   
c_uninfected = '#fde725' 
c_uptake = '#21918c'     
c_transfer = '#b5de2b'   

node_labels = [
    "Infected<br><sub>t=start</sub>", "Uninfected<br><sub>t=start</sub>", 
    "Infected<br><sub>t=end</sub>", "Uninfected<br><sub>t=end</sub>", 
    "Uptake", "Transfer"
]
node_colors = [c_infected, c_uninfected, c_infected, c_uninfected, c_uptake, c_transfer]

def hex_to_rgba(hex_col, alpha=0.3):
    hex_col = hex_col.lstrip('#')
    r, g, b = int(hex_col[0:2], 16), int(hex_col[2:4], 16), int(hex_col[4:6], 16)
    return f'rgba({r}, {g}, {b}, {alpha})'

link_colors_rgba = [
    hex_to_rgba(c_infected, 0.35), hex_to_rgba(c_infected, 0.20),
    hex_to_rgba(c_uptake, 0.35), hex_to_rgba(c_uninfected, 0.25),
    hex_to_rgba(c_uptake, 0.50), hex_to_rgba(c_transfer, 0.50),
    hex_to_rgba(c_uptake, 0.60), hex_to_rgba(c_transfer, 0.60)
]

# --- 2. Plot Construction ---
grid_conditions = [
    ('WT', 'CTRL', None), ('WT', 'INH', 'EC50'), ('WT', 'PZA', 'EC50'), ('WT', 'RIF', 'EC50'),
    ('ΔRD1', 'CTRL', None), ('WT', 'INH', 'EC99'), ('WT', 'PZA', 'EC99'), ('WT', 'RIF', 'EC99')
]

fig = make_subplots(
    rows=2, cols=4, 
    specs=[[{'type': 'sankey'}]*4]*2,
    vertical_spacing=0.12, # Space for subplot titles
    subplot_titles=[f"<b>{s} | {c} | {conc if conc else 'CTRL'}</b>" for s, c, conc in grid_conditions]
)

for i, (strain, compound, conc) in enumerate(grid_conditions):
    s, t, v = get_sankey_links(df, strain, compound, conc)
    if s:
        fig.add_trace(go.Sankey(
            node=dict(
                pad=18, thickness=12, 
                label=node_labels, color=node_colors,
                line=dict(color="black", width=0.5)
            ),
            link=dict(source=s, target=t, value=v, color=link_colors_rgba),
            textfont=dict(size=FONT_NODES, family="Helvetica Neue")
        ), row=(i // 4) + 1, col=(i % 4) + 1)

# --- 3. Final Layout & Export ---
fig.update_layout(
    title=dict(
        text="<b>Intracellular Mtb Infection Fate Transitions</b>",
        font=dict(size=FONT_TITLE),
        x=0.5, y=0.98
    ),
    width=A4_WIDTH_PX,
    height=FIG_HEIGHT_PX,
    font_family="Helvetica Neue",
    margin=dict(l=30, r=30, t=80, b=30)
)

# Update subplot title fonts
for i in fig['layout']['annotations']:
    i['font'] = dict(size=FONT_SUBPLOT)

save_path = "/Volumes/OPERA3/Nathan/data/macrohet/macrohet_results/manuscript/rebuttal_figs/final_final_final/supp/"
fig.write_html(f"{save_path}grid_sankey_viridis_fate.html")
fig.write_image(f"{save_path}grid_sankey_viridis_fate.pdf") 
fig.show()